In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pandas.tseries.offsets import DateOffset
from src.preprocessing import process_store_data
from src.features import attach_store_data, make_features, make_targets

DATA_DIR = '../datasets/rossmann-store-sales'
STORE_FILE = os.path.join(DATA_DIR, 'store.csv')
TRAIN_FILE = os.path.join(DATA_DIR, 'train.csv')
TEST_FILE = os.path.join(DATA_DIR, 'test.csv')

FORECAST_HORIZON = 6*7 # We're forecasting daily for 6 weeks into the future

LAGS = [DateOffset(days=1),   DateOffset(days=2),   DateOffset(days=7),
        DateOffset(days=14),  DateOffset(days=21),  DateOffset(months=1),
        DateOffset(months=3), DateOffset(months=6), DateOffset(years=1)]

DIFFS = [DateOffset(days=1),   DateOffset(months=1),
         DateOffset(months=3), DateOffset(months=6)]

ROLL_WINDOWS = { 7: [DateOffset(days=1), DateOffset(days=7)],
                30: [DateOffset(months=1),
                     DateOffset(months=3),
                     DateOffset(months=6)]}


# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0. This happens always


In [2]:
store_df = pd.read_csv(STORE_FILE)
store_df = process_store_data(store_df)

df_train = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)
df_train_store = attach_store_data(df_train, store_df)

# Shift Sales by 1 day per store to prevent any leakage of current-day sales into features
df_train_store = df_train_store.sort_values(['Store', 'Date'])
df_train_store['Sales_previous_day'] = df_train_store.groupby('Store')['Sales'].shift(1)

# TODO: Predict log-transformed sales?
df_features = make_features(df_train_store,
                            lags=LAGS,
                            roll_windows=ROLL_WINDOWS,
                            diffs=DIFFS)
targets = make_targets(df=df_train[['Date', 'Store', 'Sales']], horizon=FORECAST_HORIZON)

#test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
#test_df = features.attach_store_data(test_df, store_df)


C:\Users\m_kal\AppData\Local\Temp\ipykernel_21720\290780533.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)


In [3]:
pd.concat([
    df_features.dtypes,
    df_features.isna().sum()/len(df_features),
    df_features.nunique()
], axis=1).sort_values(1, ascending=False).round(2).to_csv('feature_summary.csv')

In [4]:
df_features.shape, targets.shape

((1017209, 79), (1017209, 42))

# TODOs:

- Check how the horizon is handled. Do we predict all horizons in one-go or one model per horizon?
- Check the dates/stores coming out of the cv object.
- Ensure the feature configuration is finalized. Is the way we handle categorical data correct?

In [7]:
# ── Toy dataset to sanity-check make_features ───────────────────────────────

from src.features.make_features import make_features

rng = np.random.default_rng(0)
n_stores, n_days = 2, 365*2
dates = pd.date_range("2014-01-01", periods=n_days, freq="D")
toy_rows = []
for s in range(1, n_stores + 1):
    for d in dates:
        toy_rows.append({
            "Date": d, "Store": s,
            "DayOfWeek": d.dayofweek + 1,
            "Open": 1,
            "Promo": rng.integers(0, 2),
            "Promo2": rng.integers(0, 2),
            "Promo2SinceDate": pd.NaT,
            "StateHoliday": "0",
            "SchoolHoliday": int(rng.integers(0, 2)),
            "StoreType": rng.choice(["a", "b", "c", "d"]),
            "Assortment": rng.choice(["a", "b", "c"]),
            "CompetitionDistance": rng.uniform(100, 5000),
            "CompetitionSinceDate": pd.Timestamp("2010-01-01"),
            "Sales": rng.uniform(3000, 10000),
        })

toy_df = pd.DataFrame(toy_rows)

toy_lags    = [DateOffset(days=1), DateOffset(days=7), DateOffset(months=1)]
toy_diffs   = [DateOffset(days=1), DateOffset(days=7)]
toy_windows = {7: [DateOffset(days=1)], 30: [DateOffset(months=1)]}

toy_features = make_features(toy_df.copy(), lags=toy_lags,
                              roll_windows=toy_windows, diffs=toy_diffs)

print("Shape:", toy_features.shape)
print("\nDtypes:")
print(toy_features.dtypes.to_string())
print("\nFirst row:")
print(toy_features.iloc[0].to_string())
print("\nMissing values (% of rows, non-zero only):")
missing = (toy_features.isna().sum() / len(toy_features) * 100)
print(missing[missing > 0].round(1).to_string())


Shape: (1460, 47)

Dtypes:
DayOfWeek                                 category
Open                                      category
Promo                                     category
Promo2                                    category
StateHoliday                              category
SchoolHoliday                             category
StoreType                                 category
Assortment                                category
CompetitionDistance                        float64
CompetitionSinceMonths                     float64
is_weekend                                category
is_month_start                            category
is_month_end                              category
Month                                     category
Year                                      category
Quarter                                   category
DayOfMonth_sin                             float64
DayOfMonth_cos                             float64
WeekOfYear_sin                             Float64
Week

In [8]:
toy_features.to_csv('toy_features.csv')

In [9]:
toy_df.to_csv('toy_df.csv')